# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kuteesatendojeremiah/Tendojerry-Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip install -q duckdb huggingface_hub

import os, getpass
import duckdb

# CI and power users set HF_TOKEN in the environment or Colab Secrets (🔑 icon);
# everyone else gets the safe interactive prompt. Never hardcode the token in a cell — this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")


Paste your Hugging Face READ token (hf_...): ··········
dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item, for one client, on one day** — the grain of `fact_content_daily_performance` (`report_date × client_hash_id × content_hash_id`). Dates run 2025-01-27 → 2026-06-30 (~17 months), but history depth is **not uniform**: each client's usable window starts at its own `gsc_data_start` (and separately `ga4_data_start` for engagement columns), so a global calendar window overstates what's actually available for many clients. `dim_content` and `dim_clients` sit at a coarser grain (one row per content item / per client) and describe *what* a page or client is, not *when* — they're joined onto the daily panel, never queried for a time window themselves.

In [3]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Grain violations (should be empty):")
print(grain_check)

window = con.sql(f"""
    SELECT MIN(report_date) AS earliest, MAX(report_date) AS latest,
           COUNT(DISTINCT client_hash_id) AS n_clients
    FROM {TABLES['fact_daily']}
""").df()
print("\nOverall panel window:")
print(window)

client_windows = con.sql(f"""
    SELECT MIN(gsc_data_start) AS earliest_client_start,
           MAX(gsc_data_start) AS latest_client_start,
           MIN(ga4_data_start) AS earliest_ga4_start,
           MAX(ga4_data_start) AS latest_ga4_start
    FROM {TABLES['dim_clients']}
""").df()
print("\nPer-client history start varies:")
print(client_windows)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations (should be empty):
  report_date           client_hash_id           content_hash_id  c
0  2026-06-16  client_9c26c096d6e57253  content_70b02f5039a882c2  2
1  2026-06-14  client_9c26c096d6e57253  content_01ca8d0758d6614a  2
2  2026-06-14  client_1a8bf67cad4ee525  content_3084acbc2aca184b  2
3  2026-06-18  client_b77d0d5f08f05e64  content_6b6a532cb09428b2  2
4  2026-06-17  client_1a8bf67cad4ee525  content_dd441f559f3cb4f7  2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Overall panel window:
    earliest     latest  n_clients
0 2025-01-27 2026-06-30         70

Per-client history start varies:
  earliest_client_start latest_client_start earliest_ga4_start  \
0            2025-01-27          2026-06-02         2025-10-29   

  latest_ga4_start  
0       2026-06-01  


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** — knowable before the prediction moment, safe for the model: `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (and their `_prev30`-window versions), engagement columns gated on `ga4_data_available = TRUE`, and query-mix signals from `fact_content_query_90d` (`visible_queries`, `rare_impressions_share`, `anonymized_impressions_share`, `top_query_share`) — all describe *how a page has been performing*, not the outcome we're trying to predict.

**Label / proxy** — whatever decline rule the target period defines (e.g. impressions in the last 30 days vs. the prior 30). The columns it's built from can never also be features — same trap as the starter CSV's `trend_direction`/`trend_pct`, just redefined here as `*_last30` vs `*_prev30`.

**Context** — `client_hash_id`, `content_hash_id`, `report_date` — for joining, grouping (client-level splits), and windowing only, never model inputs.

**Excluded** — `access_profile` and any FlyRank product-decision fields in `dim_clients`/`dim_content` (not observable performance signals); rows where `ga4_data_available = FALSE` should be excluded from any engagement-based feature, not filled with zero, since zero-fill there means "no data yet," not "no engagement."

In [4]:
cols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}").df()
print("fact_content_daily_performance columns:")
print(cols[["column_name", "column_type"]].to_string(index=False))

qcols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_query_90d']}").df()
print("\nfact_content_query_90d columns:")
print(qcols[["column_name", "column_type"]].to_string(index=False))

dim_client_cols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_clients']}").df()
print("\ndim_clients columns:")
print(dim_client_cols[["column_name", "column_type"]].to_string(index=False))


fact_content_daily_performance columns:
             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGIN

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Beyond the grain/window check in Section 1: `dim_clients` and `dim_content` should each be one row per key (verified below). Engagement fields are only meaningful where `ga4_data_available = TRUE` — the query below measures what share of rows that actually is, and whether it varies by client (it should, since `ga4_data_start` differs per client). `fact_content_query_90d` is a fixed 90-day window per content item; if that window overlaps the last 30 days of the panel (our likely label window), any `*_last30`-style query feature would leak — the query checks the overlap directly instead of assuming it.

In [5]:
for name in ["dim_clients", "dim_content"]:
    key = "client_hash_id" if name == "dim_clients" else "content_hash_id"
    dupes = con.sql(f"""
        SELECT {key}, COUNT(*) AS c FROM {TABLES[name]}
        GROUP BY 1 HAVING COUNT(*) > 1 LIMIT 5
    """).df()
    print(f"{name} duplicate {key} (should be empty):")
    print(dupes, "\n")

ga4_availability = con.sql(f"""
    SELECT ga4_data_available, COUNT(*) AS rows,
           COUNT(DISTINCT client_hash_id) AS n_clients
    FROM {TABLES['fact_daily']}
    GROUP BY 1
""").df()
print("GA4 availability split (zero-filled rows hide here when FALSE):")
print(ga4_availability)

missingness = con.sql(f"""
    SELECT
        AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS pct_null_position,
        AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS pct_null_impressions
    FROM {TABLES['fact_daily']}
""").df()
print("\nOverall missingness on key metrics:")
print(missingness)

overlap = con.sql(f"""
    SELECT
        (SELECT MAX(report_date) FROM {TABLES['fact_daily']}) AS panel_end,
        (SELECT MAX(report_date) - INTERVAL 90 DAY FROM {TABLES['fact_daily']}) AS query_window_start_if_aligned_to_end
""").df()
print("\nfact_content_query_90d is a fixed 90-day window per content item;")
print("compare its implicit end against the panel end to check overlap with a last-30-day label window:")
print(overlap)


dim_clients duplicate client_hash_id (should be empty):
Empty DataFrame
Columns: [client_hash_id, c]
Index: [] 

dim_content duplicate content_hash_id (should be empty):
Empty DataFrame
Columns: [content_hash_id, c]
Index: [] 



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GA4 availability split (zero-filled rows hide here when FALSE):
   ga4_data_available      rows  n_clients
0               False  46383873         60
1                <NA>  29635327         41
2                True   2816455         51


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Overall missingness on key metrics:
   pct_null_position  pct_null_impressions
0           0.632527              0.001243


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


fact_content_query_90d is a fixed 90-day window per content item;
compare its implicit end against the panel end to check overlap with a last-30-day label window:
   panel_end query_window_start_if_aligned_to_end
0 2026-06-30                           2026-04-01


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell you *why* a page's performance moved — only *that* it did. Specific limits:

- **Unbalanced panel.** History depth varies wildly by client; a third of clients have little or no usable search/analytics history. Any cross-client comparison of "trend so far" implicitly favors clients with longer histories unless windows are defined per-client.
- **GA4-only-later rows.** Rows before a client's `ga4_data_start` are zero-filled with `ga4_data_available = FALSE` — they are missing, not zero engagement. Treating them as zero would manufacture a fake engagement collapse at each client's onboarding date.
- **Window overlap risk.** `fact_content_query_90d`'s fixed 90-day window overlaps the final months of the daily panel. If a label is defined on the last 30 days, only `*_prev30`-style query features are safe — using the raw 90-day aggregate as a feature would leak label-period information into it.
- **No causal read.** Everything here is observational (search/engagement metrics), never an experiment. A correlation between a signal and later decline is not evidence that fixing the signal prevents the decline — claims must stay observed/directional, never causal.
- **`fact_content_daily_performance_sample` is the panel's last month** — it is the natural outcome window for any past→future label. Developing label logic against it means testing inside your own test set; it's fine for query mechanics, never for defining or validating a label.

In [9]:
print("Unbalanced panel — per-client history start (from Section 1):")
print(client_windows)

print("\nGA4 availability split (from Section 3) — FALSE rows are missing, not zero:")
print(ga4_availability)

print("\nQuery-table window vs. panel end (from Section 3) — checks the overlap claim above:")
print(overlap)

print((f"\nfact_content_daily_performance_sample rows: "
      f"{con.sql(f'SELECT COUNT(*) FROM {TABLES['fact_daily_sample']}').fetchone()[0]:,} "
      f"— confirm this is the panel's final month before ever using it for a label:"))
print(con.sql(f"SELECT MIN(report_date), MAX(report_date) FROM {TABLES['fact_daily_sample']}").df())

Unbalanced panel — per-client history start (from Section 1):
  earliest_client_start latest_client_start earliest_ga4_start  \
0            2025-01-27          2026-06-02         2025-10-29   

  latest_ga4_start  
0       2026-06-01  

GA4 availability split (from Section 3) — FALSE rows are missing, not zero:
   ga4_data_available      rows  n_clients
0               False  46383873         60
1                <NA>  29635327         41
2                True   2816455         51

Query-table window vs. panel end (from Section 3) — checks the overlap claim above:
   panel_end query_window_start_if_aligned_to_end
0 2026-06-30                           2026-04-01

fact_content_daily_performance_sample rows: 11,694,072 — confirm this is the panel's final month before ever using it for a label:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  min(report_date) max(report_date)
0       2026-06-01       2026-06-30


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.